In [50]:
# 1. Importar librerías

import re
import unicodedata

from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

pd.set_option(
    "display.float_format",
    lambda valor: f"{valor:,.2f}"
)

print("Librerías cargadas")

Librerías cargadas


In [2]:
# 2. Detectar la ruta del proyecto

RUTA_ACTUAL = Path.cwd().resolve()

if RUTA_ACTUAL.name.lower() == "notebooks":
    RUTA_PROYECTO = RUTA_ACTUAL.parent
elif (RUTA_ACTUAL / "datos").exists():
    RUTA_PROYECTO = RUTA_ACTUAL
else:
    RUTA_PROYECTO = RUTA_ACTUAL.parent

RUTA_INTERMEDIOS = RUTA_PROYECTO / "datos" / "intermedios"
RUTA_PROCESADOS = RUTA_PROYECTO / "datos" / "procesados"

print("🧭 Proyecto:", RUTA_PROYECTO)

🧭 Proyecto: D:\Users\LAURA PEREZ\Desktop\CIENCIA DE DATOS\PROYECTO SECOP_BARRANCABERMEJA


In [4]:
# 3. Cargar la base creada en el cuaderno 01

ARCHIVO_BASE = RUTA_INTERMEDIOS / "01_secop_ii_base_inicial.parquet"

df = pd.read_parquet(ARCHIVO_BASE)

print(f" Registros cargados: {len(df):,}")
print(f" Columnas: {df.shape[1]}")

 Registros cargados: 37,574
 Columnas: 48


In [5]:
# 4. Revisar columnas disponibles

pd.set_option("display.max_rows", 100)

display(
    pd.DataFrame({
        "columna": df.columns
    })
)

,columna
0,nombre_entidad
1,nit_entidad
2,departamento
3,ciudad
4,localizaci_n
5,orden
6,sector
7,rama
8,entidad_centralizada
9,proceso_de_compra


In [7]:
# 5. Verificar campos esenciales

COLUMNAS_ESENCIALES = [
    "id_contrato",
    "nombre_entidad",
    "nit_entidad",
    "tipo_de_contrato",
    "modalidad_de_contratacion",
    "fecha_de_firma",
    "fecha_de_inicio_del_contrato",
    "fecha_de_fin_del_contrato",
    "tipodocproveedor",
    "documento_proveedor",
    "proveedor_adjudicado",
    "valor_del_contrato"
]

faltantes = [
    columna
    for columna in COLUMNAS_ESENCIALES
    if columna not in df.columns
]

if faltantes:
    raise ValueError(f"Faltan columnas esenciales: {faltantes}")

print("Campos esenciales verificados")

Campos esenciales verificados


In [8]:
# 6. Crear copia de trabajo

base = df.copy()

print(f"Base de trabajo: {len(base):,} registros")

Base de trabajo: 37,574 registros


In [9]:
# 7. Función para normalizar texto

def normalizar_texto(valor):
    if pd.isna(valor):
        return ""

    valor = str(valor).strip().upper()
    valor = unicodedata.normalize("NFKD", valor)
    valor = "".join(
        caracter
        for caracter in valor
        if not unicodedata.combining(caracter)
    )

    valor = re.sub(r"\s+", " ", valor)

    return valor

In [11]:
# 8. Normalizar campos de texto

base["tipo_contrato_norm"] = (
    base["tipo_de_contrato"]
    .map(normalizar_texto)
)

base["modalidad_norm"] = (
    base["modalidad_de_contratacion"]
    .map(normalizar_texto)
)

base["tipo_documento_norm"] = (
    base["tipodocproveedor"]
    .map(normalizar_texto)
)

base["proveedor_norm"] = (
    base["proveedor_adjudicado"]
    .map(normalizar_texto)
)

print("Textos normalizados")

Textos normalizados


In [12]:
# 9. Detectar la columna de justificación

columnas_justificacion = [
    columna
    for columna in base.columns
    if "justificacion" in columna.lower()
]

print(columnas_justificacion)

['justificacion_modalidad_de']


In [13]:
# 10. Crear justificación normalizada

if columnas_justificacion:
    COLUMNA_JUSTIFICACION = columnas_justificacion[0]

    base["justificacion_norm"] = (
        base[COLUMNA_JUSTIFICACION]
        .map(normalizar_texto)
    )

else:
    COLUMNA_JUSTIFICACION = None
    base["justificacion_norm"] = ""

print("Columna usada:", COLUMNA_JUSTIFICACION)

Columna usada: justificacion_modalidad_de


In [15]:
# 11. Normalizar documento del proveedor

base["documento_normalizado"] = (
    base["documento_proveedor"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"[^A-Z0-9]", "", regex=True)
)

base["documento_normalizado"] = (
    base["documento_normalizado"]
    .replace("", pd.NA)
)

print("Documentos normalizados")

Documentos normalizados


In [16]:
# 12. Crear llave única del proveedor

base["proveedor_llave"] = np.where(
    base["documento_normalizado"].notna(),
    base["documento_normalizado"],
    base["proveedor_norm"]
)

print(
    f" Proveedores identificados: "
    f"{base['proveedor_llave'].nunique():,}"
)

 Proveedores identificados: 11,328


In [17]:
# 13. Clasificar persona natural o jurídica

PATRON_NATURAL = (
    r"CEDULA DE CIUDADANIA|CEDULA DE EXTRANJERIA|"
    r"PASAPORTE|PERMISO POR PROTECCION TEMPORAL"
)

PATRON_JURIDICA = (
    r"NIT|REGISTRO UNICO TRIBUTARIO"
)

base["tipo_proveedor"] = np.select(
    [
        base["tipo_documento_norm"].str.contains(
            PATRON_NATURAL,
            regex=True,
            na=False
        ),
        base["tipo_documento_norm"].str.contains(
            PATRON_JURIDICA,
            regex=True,
            na=False
        )
    ],
    [
        "Persona natural",
        "Persona jurídica"
    ],
    default="Por revisar"
)

base["tipo_proveedor"].value_counts(dropna=False)

tipo_proveedor
Persona natural     35029
Persona jurídica     2490
Por revisar            55
Name: count, dtype: int64

In [18]:
# 14. Revisar tipos de documento

display(
    base[
        [
            "tipodocproveedor",
            "tipo_documento_norm",
            "tipo_proveedor"
        ]
    ]
    .value_counts()
    .reset_index(name="registros")
    .head(30)
)

,tipodocproveedor,tipo_documento_norm,tipo_proveedor,registros
0,Cédula de Ciudadanía,CEDULA DE CIUDADANIA,Persona natural,34994
1,NIT,NIT,Persona jurídica,2490
2,Otro,OTRO,Por revisar,45
3,Cédula de Extranjería,CEDULA DE EXTRANJERIA,Persona natural,25
4,Permiso por Protección Temporal,PERMISO POR PROTECCION TEMPORAL,Persona natural,10
5,No Definido,NO DEFINIDO,Por revisar,9
6,Registro Civil,REGISTRO CIVIL,Por revisar,1


In [19]:
# 15. Identificar contratos de prestación de servicios

base["es_prestacion_servicios"] = (
    base["tipo_contrato_norm"]
    .str.contains(
        "PRESTACION DE SERVICIOS",
        regex=False,
        na=False
    )
)

base["es_prestacion_servicios"].value_counts()

es_prestacion_servicios
True     33597
False     3977
Name: count, dtype: int64

In [20]:
# 16. Identificar servicios profesionales y apoyo a la gestión

PATRON_CPS = (
    r"PRESTACION DE SERVICIOS PROFESIONALES|"
    r"SERVICIOS PROFESIONALES|"
    r"APOYO A LA GESTION"
)

base["justificacion_cps"] = (
    base["justificacion_norm"]
    .str.contains(
        PATRON_CPS,
        regex=True,
        na=False
    )
)

base["justificacion_cps"].value_counts()

justificacion_cps
True     32582
False     4992
Name: count, dtype: int64

In [21]:
# 17. Clasificar CPS

condiciones_cps = [
    (
        base["es_prestacion_servicios"]
        & base["tipo_proveedor"].eq("Persona natural")
        & base["justificacion_cps"]
    ),
    (
        base["es_prestacion_servicios"]
        & base["tipo_proveedor"].eq("Persona natural")
    )
]

categorias_cps = [
    "CPS confirmado",
    "CPS probable"
]

base["clasificacion_cps"] = np.select(
    condiciones_cps,
    categorias_cps,
    default="No CPS"
)

base["clasificacion_cps"].value_counts()

clasificacion_cps
CPS confirmado    32392
No CPS             4715
CPS probable        467
Name: count, dtype: int64

In [22]:
# 18. Crear indicador general de CPS persona natural

base["es_cps_persona_natural"] = (
    base["clasificacion_cps"]
    .isin(
        [
            "CPS confirmado",
            "CPS probable"
        ]
    )
)

base["es_cps_persona_natural"].value_counts()

es_cps_persona_natural
True     32859
False     4715
Name: count, dtype: int64

In [23]:
# 19. Identificar empresas prestadoras de servicios

base["es_servicio_empresa"] = (
    base["es_prestacion_servicios"]
    & base["tipo_proveedor"].eq("Persona jurídica")
)

base["es_servicio_empresa"].value_counts()

es_servicio_empresa
False    36856
True       718
Name: count, dtype: int64

In [24]:
# 20. Clasificar familias de contratos

condiciones_familia = [
    base["es_cps_persona_natural"],

    base["es_servicio_empresa"],

    base["tipo_contrato_norm"].str.contains(
        "OBRA",
        na=False
    ),

    base["tipo_contrato_norm"].str.contains(
        "CONSULTORIA|INTERVENTORIA",
        regex=True,
        na=False
    ),

    base["tipo_contrato_norm"].str.contains(
        "SUMINISTRO|COMPRAVENTA",
        regex=True,
        na=False
    ),

    base["tipo_contrato_norm"].str.contains(
        "ARRENDAMIENTO",
        na=False
    ),

    (
        base["tipo_contrato_norm"].str.contains(
            "CONVENIO|INTERADMINISTRATIVO",
            regex=True,
            na=False
        )
        |
        base["modalidad_norm"].str.contains(
            "INTERADMINISTRATIVO",
            na=False
        )
    )
]

categorias_familia = [
    "CPS persona natural",
    "Servicios empresa",
    "Obra",
    "Consultoría e interventoría",
    "Bienes y suministros",
    "Arrendamiento",
    "Convenios e interadministrativos"
]

base["familia_contrato"] = np.select(
    condiciones_familia,
    categorias_familia,
    default="Otros"
)

base["familia_contrato"].value_counts()

familia_contrato
CPS persona natural            32859
Otros                           3234
Servicios empresa                718
Bienes y suministros             397
Arrendamiento                    174
Obra                              98
Consultoría e interventoría       94
Name: count, dtype: int64

In [25]:
# 21. Identificar mínima cuantía

base["es_minima_cuantia"] = (
    base["modalidad_norm"]
    .str.contains(
        "MINIMA CUANTIA",
        na=False
    )
)

base["es_minima_cuantia"].value_counts()

es_minima_cuantia
False    37148
True       426
Name: count, dtype: int64

In [27]:
# 22. Convertir fechas

COLUMNAS_FECHA = [
    "fecha_de_firma",
    "fecha_de_inicio_del_contrato",
    "fecha_de_fin_del_contrato"
]

for columna in COLUMNAS_FECHA:
    base[columna] = pd.to_datetime(
        base[columna],
        errors="coerce"
    )

print("Fechas convertidas")

Fechas convertidas


In [29]:
# 23. Calcular duración contractual

base["duracion_dias"] = (
    base["fecha_de_fin_del_contrato"]
    - base["fecha_de_inicio_del_contrato"]
).dt.days

base.loc[
    base["duracion_dias"] < 0,
    "duracion_dias"
] = np.nan

base["duracion_meses"] = (
    base["duracion_dias"] / 30.44
)

print("Duración calculada")

Duración calculada


In [30]:
# 24. Convertir valores monetarios

base["valor_contrato_num"] = pd.to_numeric(
    base["valor_del_contrato"],
    errors="coerce"
)

base.loc[
    base["valor_contrato_num"] < 0,
    "valor_contrato_num"
] = np.nan

print(" Valores convertidos")

 Valores convertidos


In [31]:
# 25. Calcular valor mensual equivalente

base["valor_mensual_equivalente"] = np.where(
    base["duracion_meses"] > 0,
    base["valor_contrato_num"] / base["duracion_meses"],
    np.nan
)

print("Valor mensual equivalente calculado")

Valor mensual equivalente calculado


In [32]:
# 26. Clasificar duración de los contratos

base["rango_duracion"] = pd.cut(
    base["duracion_meses"],
    bins=[
        -np.inf,
        3,
        4,
        6,
        9,
        12,
        np.inf
    ],
    labels=[
        "Hasta 3 meses",
        "Más de 3 hasta 4",
        "Más de 4 hasta 6",
        "Más de 6 hasta 9",
        "Más de 9 hasta 12",
        "Más de 12 meses"
    ]
)

base["rango_duracion"].value_counts(
    dropna=False
).sort_index()

rango_duracion
Hasta 3 meses        18154
Más de 3 hasta 4      7471
Más de 4 hasta 6      9189
Más de 6 hasta 9      1697
Más de 9 hasta 12      597
Más de 12 meses        133
NaN                    333
Name: count, dtype: int64

In [33]:
# 27. Identificar administración para contratos de la Alcaldía

base["alcalde"] = np.select(
    [
        (
            base["es_alcaldia"]
            & base["fecha_referencia"].between(
                "2020-01-01",
                "2023-12-31 23:59:59"
            )
        ),
        (
            base["es_alcaldia"]
            & base["fecha_referencia"].between(
                "2024-01-01",
                "2026-09-06 23:59:59"
            )
        )
    ],
    [
        "Alfonso Eljach",
        "Jonathan Vásquez"
    ],
    default="No aplica"
)

base["alcalde"].value_counts()

alcalde
Jonathan Vásquez    14243
Alfonso Eljach      13687
No aplica            9644
Name: count, dtype: int64

In [34]:
# 28. Crear año de gobierno

base["anio_gobierno"] = np.select(
    [
        base["alcalde"].eq("Alfonso Eljach"),
        base["alcalde"].eq("Jonathan Vásquez")
    ],
    [
        base["anio_referencia"] - 2019,
        base["anio_referencia"] - 2023
    ],
    default=np.nan
)

base["anio_gobierno"] = pd.to_numeric(
    base["anio_gobierno"],
    errors="coerce"
).astype("Int64")

display(
    base[
        [
            "alcalde",
            "anio_referencia",
            "anio_gobierno"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "alcalde",
            "anio_referencia"
        ]
    )
)

,alcalde,anio_referencia,anio_gobierno
0,Alfonso Eljach,2020,1
90,Alfonso Eljach,2021,2
3653,Alfonso Eljach,2022,3
10794,Alfonso Eljach,2023,4
17042,Jonathan Vásquez,2024,1
23589,Jonathan Vásquez,2025,2
31020,Jonathan Vásquez,2026,3
3,No aplica,2020,<NA>
80,No aplica,2021,<NA>
3594,No aplica,2022,<NA>


In [37]:
# 29. Detectar duplicados exactos

# Algunas columnas contienen diccionarios, que pandas no puede hashear directamente.
base_comparable = base.map(
    lambda valor: tuple(sorted(valor.items()))
    if isinstance(valor, dict)
    else valor
)

duplicados_exactos = base.loc[
    base_comparable.duplicated(
        keep=False
    )
].copy()

print(
    f"⚠️ Filas en duplicados exactos: "
    f"{len(duplicados_exactos):,}"
)

⚠️ Filas en duplicados exactos: 0


In [39]:
# 30. Eliminar únicamente duplicados completamente idénticos

base = (
    base.loc[
        ~base_comparable.duplicated()
    ]
    .reset_index(drop=True)
)

print(
    f"Registros después de duplicados exactos: "
    f"{len(base):,}"
)

Registros después de duplicados exactos: 37,574


In [40]:
# 31. Revisar ID de contrato repetidos

duplicados_id = base[
    base["id_contrato"].notna()
    & base["id_contrato"].duplicated(
        keep=False
    )
].copy()

print(
    f"⚠️ Filas con ID contrato repetido: "
    f"{len(duplicados_id):,}"
)

print(
    f"⚠️ ID diferentes repetidos: "
    f"{duplicados_id['id_contrato'].nunique():,}"
)

⚠️ Filas con ID contrato repetido: 0
⚠️ ID diferentes repetidos: 0


In [41]:
# 32. Crear puntaje de completitud

COLUMNAS_COMPLETITUD = [
    "id_contrato",
    "documento_normalizado",
    "proveedor_adjudicado",
    "tipo_de_contrato",
    "modalidad_de_contratacion",
    "fecha_de_firma",
    "fecha_de_inicio_del_contrato",
    "fecha_de_fin_del_contrato",
    "valor_contrato_num"
]

base["puntaje_completitud"] = (
    base[COLUMNAS_COMPLETITUD]
    .notna()
    .sum(axis=1)
)

In [43]:
# 33. Crear base de un registro por contrato

base_contratos = (
    base
    .sort_values(
        "puntaje_completitud",
        ascending=False
    )
    .drop_duplicates(
        subset="id_contrato",
        keep="first"
    )
    .reset_index(drop=True)
)

print(
    f" Contratos únicos: "
    f"{len(base_contratos):,}"
)

 Contratos únicos: 37,574


In [44]:
# 34. Crear universo CPS de personas naturales

cps = base_contratos[
    base_contratos["es_cps_persona_natural"]
].copy()

print(f" CPS identificados: {len(cps):,}")

print(
    f"Personas únicas en CPS: "
    f"{cps['proveedor_llave'].nunique():,}"
)

 CPS identificados: 32,859
Personas únicas en CPS: 9,944


In [46]:
# 35. Crear universo de empresas

empresas = base_contratos[
    base_contratos["tipo_proveedor"].eq(
        "Persona jurídica"
    )
].copy()

print(
    f" Contratos con personas jurídicas: "
    f"{len(empresas):,}"
)

print(
    f"Empresas únicas: "
    f"{empresas['proveedor_llave'].nunique():,}"
)

 Contratos con personas jurídicas: 2,490
Empresas únicas: 696


In [51]:
# 36. Revisar clasificación general

resumen_familias = (
    base_contratos
    .groupby(
        "familia_contrato",
        dropna=False
    )
    .agg(
        contratos=("id_contrato", "nunique"),
        proveedores=("proveedor_llave", "nunique"),
        valor_total=("valor_contrato_num", "sum")
    )
    .reset_index()
    .sort_values(
        "contratos",
        ascending=False
    )
)

display(resumen_familias)

,familia_contrato,contratos,proveedores,valor_total
2,CPS persona natural,32859,9944,"398,610,382,773.97"
5,Otros,3234,1332,"665,849,414,302.57"
6,Servicios empresa,718,258,"495,758,604,308.63"
1,Bienes y suministros,397,143,"89,515,102,499.97"
0,Arrendamiento,174,66,"21,322,197,091.38"
4,Obra,98,76,"271,742,091,652.33"
3,Consultoría e interventoría,94,63,"46,724,857,774.56"


In [54]:
# 37. Revisar CPS por alcalde

cps_alcaldia = cps.loc[
    cps["es_alcaldia"]
].copy()

texto_proceso = cps_alcaldia["descripcion_del_proceso"].map(
    normalizar_texto
)

menciona_profesional = texto_proceso.str.contains(
    "PROFESIONAL",
    na=False
)
menciona_apoyo = texto_proceso.str.contains(
    "APOYO A LA GESTION",
    na=False
)

cps_alcaldia["tipo_cps"] = np.select(
    [
        menciona_profesional & ~menciona_apoyo,
        menciona_apoyo & ~menciona_profesional
    ],
    [
        "Profesional",
        "Apoyo a la gestión"
    ],
    default="Sin clasificar"
)

resumen_cps_alcaldes = (
    cps_alcaldia
    .groupby(
        "alcalde"
    )
    .agg(
        contratos_cps=("id_contrato", "nunique"),
        personas=("proveedor_llave", "nunique"),
        valor_total=("valor_contrato_num", "sum"),
        duracion_mediana_meses=("duracion_meses", "median"),
        valor_mensual_mediano_profesional=(
            "valor_mensual_equivalente",
            lambda valores: valores.loc[
                cps_alcaldia.loc[valores.index, "tipo_cps"].eq(
                    "Profesional"
                )
            ].median()
        ),
        valor_mensual_mediano_apoyo_gestion=(
            "valor_mensual_equivalente",
            lambda valores: valores.loc[
                cps_alcaldia.loc[valores.index, "tipo_cps"].eq(
                    "Apoyo a la gestión"
                )
            ].median()
        )
    )
    .reset_index()
)

display(resumen_cps_alcaldes)

,alcalde,contratos_cps,personas,valor_total,duracion_mediana_meses,valor_mensual_mediano_profesional,valor_mensual_mediano_apoyo_gestion
0,Alfonso Eljach,12816,4996,"141,932,482,194.99",3.88,"3,551,333.33","2,007,032.97"
1,Jonathan Vásquez,13516,5320,"167,402,346,789.34",2.99,"4,058,666.67","2,565,168.54"


In [55]:
# 38. Revisar valores faltantes en CPS

calidad_cps = pd.DataFrame({
    "variable": [
        "documento",
        "fecha inicio",
        "fecha fin",
        "duración",
        "valor contrato",
        "valor mensual"
    ],

    "faltantes": [
        cps["documento_normalizado"].isna().sum(),
        cps["fecha_de_inicio_del_contrato"].isna().sum(),
        cps["fecha_de_fin_del_contrato"].isna().sum(),
        cps["duracion_meses"].isna().sum(),
        cps["valor_contrato_num"].isna().sum(),
        cps["valor_mensual_equivalente"].isna().sum()
    ]
})

calidad_cps["porcentaje"] = (
    calidad_cps["faltantes"]
    / len(cps)
    * 100
).round(2)

display(calidad_cps)

,variable,faltantes,porcentaje
0,documento,0,0.00
1,fecha inicio,285,0.87
2,fecha fin,0,0.00
3,duración,289,0.88
4,valor contrato,0,0.00
5,valor mensual,363,1.10


In [56]:
# 39. Guardar reporte de duplicados

RUTA_DUPLICADOS = (
    RUTA_INTERMEDIOS
    / "02_contratos_id_duplicado.csv"
)

duplicados_id.to_csv(
    RUTA_DUPLICADOS,
    index=False,
    encoding="utf-8-sig"
)

print("Duplicados guardados")

Duplicados guardados


In [57]:
# 40. Guardar base maestra clasificada

RUTA_BASE_MAESTRA = (
    RUTA_INTERMEDIOS
    / "02_base_maestra_clasificada.parquet"
)

base_contratos.to_parquet(
    RUTA_BASE_MAESTRA,
    index=False
)

print("Base maestra guardada")

Base maestra guardada


In [58]:
# 41. Guardar universo CPS

RUTA_CPS = (
    RUTA_PROCESADOS
    / "02_cps_personas_naturales.parquet"
)

cps.to_parquet(
    RUTA_CPS,
    index=False
)

print("Base CPS guardada")

Base CPS guardada


In [59]:
# 42. Guardar universo de empresas

RUTA_EMPRESAS = (
    RUTA_PROCESADOS
    / "02_empresas_contratistas.parquet"
)

empresas.to_parquet(
    RUTA_EMPRESAS,
    index=False
)

print("Base de empresas guardada")

Base de empresas guardada


In [60]:
# 43. Guardar resúmenes de control

resumen_familias.to_csv(
    RUTA_INTERMEDIOS / "02_resumen_familias_contrato.csv",
    index=False,
    encoding="utf-8-sig"
)

resumen_cps_alcaldes.to_csv(
    RUTA_INTERMEDIOS / "02_resumen_cps_alcaldes.csv",
    index=False,
    encoding="utf-8-sig"
)

calidad_cps.to_csv(
    RUTA_INTERMEDIOS / "02_calidad_cps.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Resúmenes guardados")

Resúmenes guardados


In [61]:
# 44. Control final

print("=" * 60)

print(f"Base original: {len(df):,}")
print(f" Contratos únicos: {len(base_contratos):,}")
print(f" CPS persona natural: {len(cps):,}")
print(f"Personas únicas CPS: {cps['proveedor_llave'].nunique():,}")
print(f" Contratos con empresas: {len(empresas):,}")
print(f" Empresas únicas: {empresas['proveedor_llave'].nunique():,}")

print("=" * 60)

Base original: 37,574
 Contratos únicos: 37,574
 CPS persona natural: 32,859
Personas únicas CPS: 9,944
 Contratos con empresas: 2,490
 Empresas únicas: 696
